# Lección 8.2 — K-means: repartir carritos de helado en la playa

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 8.1 (estructura sin etiquetas), Lección 5.3 (KNN y distancias), Lección 2.3 (escalado)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo K-means forma grupos alrededor de **centroides**.
- Ejecutar **a mano** el algoritmo: asignar → mover centros → repetir.
- Elegir el número de grupos `k` con el **método del codo** y la **silueta**.
- Predecir cuándo K-means fracasa (formas raras, escalas distintas, atípicos) y qué hacer.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: carritos de helado en la playa 🍦

Tienes **3 carritos** de helado y una playa llena de bañistas. ¿Dónde los pones para que **cada persona camine lo menos posible** hasta el carrito más cercano?

Una estrategia sencilla:
1. Pon los 3 carritos en cualquier lugar.
2. Cada bañista va al carrito **más cercano**.
3. Cada carrito se mueve al **centro** de "sus" clientes.
4. Repite 2 y 3 hasta que los carritos ya no se muevan.

Eso es **K-means** completo. La **K** es el número de carritos (grupos) y *means* (medias) son los **centros**: el promedio de los puntos de cada grupo. A ese centro se le llama **centroide**.

### ✋ Ejercicio 1 — Calentamiento
En la estrategia anterior, ¿qué tienes que decidir **tú** antes de empezar y qué **descubre** el algoritmo?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Tú decides:** cuántos carritos (`k`) y dónde ponerlos al principio (la **inicialización**). Son los dos hiperparámetros clave.
- **El algoritmo descubre:** la posición final de cada carrito (los **centroides**) y qué bañista va a cuál (los **grupos**).

Fíjate en que `k` lo eliges tú: K-means **no sabe** cuántos grupos hay. En la sección 5 veremos cómo elegirlo.
</details>

---
## 2. 📖 El concepto: centroides e inercia

| Término | Qué es | En la analogía |
|---|---|---|
| **k** | Número de grupos | Número de carritos |
| **Centroide** | El promedio de los puntos del grupo | Posición del carrito |
| **Asignación** | Cada punto pertenece al centroide más cercano | Cada bañista va al carrito más cercano |
| **Inercia** | Suma de las distancias (al cuadrado) de cada punto a su centroide | Cuánto caminan **todos** los bañistas en total |

K-means busca la posición de los carritos que **minimiza la inercia**: grupos lo más **compactos** posible. Es su "función de pérdida" (Lección 3.2).

### ✋ Ejercicio 2 — K-means a mano
Seis bañistas están en una playa recta, en las posiciones (metros):

```
1   2   3                          10  11  12
```

Ponemos 2 carritos al principio en las posiciones **1** y **2** (una mala inicialización a propósito). Haz las iteraciones a mano:

1. **Asignar:** ¿qué bañistas van a cada carrito?
2. **Mover:** ¿a dónde se mueve cada carrito (promedio de sus bañistas)?
3. Repite hasta que no cambie nada. ¿Dónde terminan los carritos?

✍️ *Mis cálculos:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Iteración 1**
- Asignar: el carrito en 1 se queda solo con el bañista en **1**; el carrito en 2 se queda con **2, 3, 10, 11, 12** (todos están más cerca de 2 que de 1).
- Mover: carrito A → 1; carrito B → (2+3+10+11+12)/5 = **7.6**.

**Iteración 2**
- Asignar: 1, 2 y 3 están más cerca de 1 que de 7.6 → A. 10, 11 y 12 → B.
- Mover: A → (1+2+3)/3 = **2**; B → (10+11+12)/3 = **11**.

**Iteración 3:** nadie cambia de carrito → **terminamos**. Carritos en **2** y **11**.

Comprueba con la computadora en la celda siguiente.
</details>

In [ ]:
playa = np.array([1, 2, 3, 10, 11, 12]).reshape(-1, 1)
inicio = np.array([[1], [2]])            # carritos iniciales (mala inicialización a propósito)

km = KMeans(n_clusters=2, init=inicio, n_init=1).fit(playa)
print("Carritos finales :", km.cluster_centers_.ravel())
print("Grupo de cada uno:", km.labels_)
print("Inercia          :", km.inertia_)

> 📝 La inercia vale **4**: cada grupo tiene dos bañistas a 1 metro de su carrito (1² + 1² por grupo). Ningún otro reparto de 2 carritos hace caminar menos a esta playa.

---
## 3. 📖 Cómo funciona por dentro: verlo paso a paso

Ahora en 2D, con 300 puntos y 3 grupos. La función `un_paso` hace exactamente lo que hiciste a mano: **asignar** y **mover**. Empezamos con los 3 centroides amontonados dentro del grupo de la izquierda (una pésima inicialización a propósito) para que se note el viaje.

In [ ]:
X, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.9, random_state=7)

def un_paso(X, centros):
    distancias = np.linalg.norm(X[:, None, :] - centros[None, :, :], axis=2)
    grupo = distancias.argmin(axis=1)                                  # 1) asignar al más cercano
    nuevos = np.array([X[grupo == j].mean(axis=0) for j in range(len(centros))])   # 2) mover al promedio
    inercia = (distancias[np.arange(len(X)), grupo] ** 2).sum()
    return grupo, nuevos, inercia

centros = np.array([[-9.0, 5.0], [-8.0, 6.0], [-8.0, 5.0]])     # los 3 amontonados en un mismo grupo
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for i, ax in enumerate(axes):
    grupo, nuevos, inercia = un_paso(X, centros)
    ax.scatter(*X.T, c=grupo, cmap="viridis", s=12)
    ax.scatter(*centros.T, c="red", marker="X", s=200, edgecolor="k")
    ax.set_title(f"Iteración {i + 1} · inercia {inercia:,.0f}", fontsize=10)
    centros = nuevos
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 3
Observa las 5 gráficas (las ✖️ rojas son los centroides **antes** de moverse en esa iteración).
1. ¿Qué le pasa a la inercia de una iteración a la siguiente?
2. ¿Puede **subir** alguna vez?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Baja** en cada iteración (≈38,000 → 7,000 → 5,000 → 735 → 464) hasta estabilizarse: los centroides viajan desde el grupo de la izquierda hasta el centro de cada grupo. A partir de la 5.ª iteración ya nadie cambia de grupo.
2. **No.** Cada paso solo puede mejorar o dejar igual: asignar cada punto a su centro más cercano no aumenta las distancias, y mover el centro al promedio tampoco. Por eso K-means **siempre termina**. Pero que termine no significa que encuentre la **mejor** solución… (sigue leyendo).
</details>

---
## 4. 📖 ¿Qué pasa si… empiezas en mal lugar?

K-means baja la inercia "cuesta abajo" como el excursionista de la Lección 3.3, y puede quedarse atrapado en un **mínimo local**: una solución estable pero mala (p. ej. dos carritos compartiendo un grupo y un carrito cubriendo dos grupos).

Por eso scikit-learn tiene dos defensas:
- **`init="k-means++"`** (por defecto): elige los centros iniciales **separados** entre sí, en vez de al azar.
- **`n_init`**: repite todo el algoritmo varias veces con distintos inicios y se queda con la de **menor inercia**.

### 🔮 Ejercicio 4
Usaremos 8 grupos y la inicialización **más ingenua** (`init="random"`, `n_init=1`), con 6 semillas distintas. ¿Darán todas la misma inercia?

✍️ *Mi apuesta:*

In [ ]:
X8, _ = make_blobs(n_samples=800, centers=8, cluster_std=0.7, random_state=3)

for semilla in range(6):
    km = KMeans(n_clusters=8, init="random", n_init=1, random_state=semilla).fit(X8)
    print(f"semilla {semilla}: inercia = {km.inertia_:8.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Las mejores semillas llegan a una inercia de ≈**760–770**, pero otras se quedan atrapadas en mínimos locales con casi el doble (≈1,300) o más del triple (≈2,500): dos centroides comparten un grupo mientras otro centroide cubre dos grupos. Mismo algoritmo, mismos datos: la diferencia es **solo dónde empezó**.
</details>

### ✏️ Ejercicio 5 — Las dos defensas
Cambia la celda de abajo: primero `INIT = "k-means++"` con `N_INIT = 1`; después `INIT = "random"` con `N_INIT = 10`. ¿Cuál de las dos defensas arregla el problema?

In [ ]:
INIT = "random"     # ✏️ prueba "k-means++"
N_INIT = 1          # ✏️ prueba 10

for semilla in range(6):
    km = KMeans(n_clusters=8, init=INIT, n_init=N_INIT, random_state=semilla).fit(X8)
    print(f"semilla {semilla}: inercia = {km.inertia_:8.1f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `k-means++` con `n_init=1`: desaparecen los desastres; todas las semillas quedan entre ≈731 y ≈771.
- `random` con `n_init=10`: también desaparecen los desastres (≈731–781), porque de 10 intentos al menos uno sale decente.

¡Ojo! La mejor solución es ≈**731**, mejor aún que la mejor de la tabla anterior. Lo más seguro es **combinar ambas**: con `INIT = "k-means++"` y `N_INIT = 10`, las 6 semillas dan ≈731. Es la configuración por defecto de scikit-learn. Repetir cuesta tiempo de cómputo, pero K-means es tan rápido que casi siempre vale la pena.
</details>

### 📖 Las otras debilidades: K-means imagina "pelotas"
Como cada punto va al centro **más cercano**, K-means siempre parte el espacio en regiones "redondeadas" alrededor de cada centro. Funciona de maravilla si los grupos son **pelotas de tamaño parecido**. Si no:

### 🔮 Ejercicio 6
Abajo hay tres conjuntos con 2 o 3 grupos evidentes a simple vista: **lunas**, **grupos alargados** y **grupos de tamaños muy distintos**. Le daremos a K-means el `k` correcto en los tres. ¿En cuáles crees que acertará?

✍️ *Mi apuesta:*

In [ ]:
X_lunas, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
X_alarg, _ = make_blobs(n_samples=300, centers=3, random_state=170)
X_alarg = X_alarg @ np.array([[0.6, -0.64], [-0.41, 0.85]])        # estiramos los grupos
X_tam = np.vstack([np.random.default_rng(0).normal([0, 0], 2.5, (400, 2)),
                   np.random.default_rng(1).normal([5.5, 0], 0.4, (30, 2))])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, datos, k) in zip(axes, [("Lunas", X_lunas, 2), ("Alargados", X_alarg, 3),
                                          ("Tamaños distintos", X_tam, 2)]):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(datos)
    ax.scatter(*datos.T, c=km.labels_, cmap="viridis", s=12)
    ax.scatter(*km.cluster_centers_.T, c="red", marker="X", s=200, edgecolor="k")
    ax.set_title(f"{nombre} (k={k})")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**En ninguno lo hace bien:**
- **Lunas:** corta cada luna por la mitad con una línea recta. No entiende formas curvas.
- **Alargados:** mezcla pedazos de grupos vecinos, porque busca regiones redondas y los grupos son "salchichas".
- **Tamaños distintos:** el grupo pequeño y compacto (30 puntos) termina con un "grupo" de casi 200: se queda con buena parte del grupo grande y disperso. La frontera queda a medio camino entre los centros, no donde termina cada grupo.

Para formas arbitrarias y ruido existe **DBSCAN** (Lección 8.3). Para grupos alargados o de distinto tamaño, un **modelo de mezclas gaussianas** (`GaussianMixture`) suele ir mejor (ver "Para profundizar").
</details>

---
## 5. 📖 Hiperparámetros: ¿cuántos carritos pongo?

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_clusters` (k) | Número de grupos | Grupos más pequeños y compactos; la inercia **siempre** baja | Grupos más grandes y mezclados | Codo + silueta + negocio (2–10) |
| `init` | Cómo se eligen los centros iniciales | — | — | `"k-means++"` |
| `n_init` | Cuántas veces repite con inicios distintos | Más estable, más lento | Riesgo de mínimo local | 10 (o `"auto"`) |
| `max_iter` | Máximo de iteraciones por intento | — | Puede cortar antes de converger | 300 (rara vez se toca) |
| `random_state` | Semilla de la inicialización | — | — | Fíjala para poder repetir resultados |

### 🧠 La trampa de la inercia
Si eliges `k` buscando la **menor inercia**, siempre ganará el `k` más grande: con un carrito por bañista, nadie camina (inercia 0). Por eso no se minimiza la inercia, se busca un **codo**.

### Herramienta 1: el método del codo
Dibujas la inercia para k = 1, 2, 3… Al principio cada carrito extra ahorra **muchísimo** camino; a partir del número "real" de grupos, cada carrito extra ahorra **poco**. El punto donde la curva se dobla es el **codo**.

### 🔮 Ejercicio 7
Generamos datos con **4 grupos** (no los ves todavía). ¿En qué k esperas el codo?

✍️ *Mi apuesta:*

In [ ]:
X4, _ = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=8)

K = range(1, 11)
inercias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X4).inertia_ for k in K]

plt.plot(K, inercias, "o-")
plt.xlabel("k (número de grupos)"); plt.ylabel("Inercia"); plt.grid(alpha=.3)
plt.title("Método del codo"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

El codo está en **k = 4**: de 1 a 4 la inercia cae en picada; de 4 en adelante baja muy despacio. Cada grupo extra después de 4 solo parte en dos un grupo que ya era compacto.

En datos reales el codo rara vez es tan claro. Por eso se usa una segunda herramienta.
</details>

### Herramienta 2: la silueta
Para **cada punto** la silueta compara:
- **a** = distancia promedio a los puntos de **su propio** grupo (¿qué tan apretado está?).
- **b** = distancia promedio a los puntos del **grupo vecino** más cercano (¿qué tan lejos está del otro?).

Silueta = (b − a) / máx(a, b). Va de **−1 a 1**:
- **≈ 1**: el punto está en su grupo y lejos de los demás. 😀
- **≈ 0**: está en la frontera entre dos grupos. 😐
- **< 0**: probablemente está en el grupo equivocado. 😟

El **promedio** de todos los puntos resume la calidad del agrupamiento. A diferencia de la inercia, **no** mejora siempre al subir k: se busca su **máximo**.

### 🔮 Ejercicio 8
¿Qué k tendrá la mayor silueta promedio para los mismos datos? (Ojo: la silueta no existe para k = 1.)

✍️ *Mi apuesta:*

In [ ]:
for k in range(2, 9):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X4)
    s = silhouette_score(X4, etiquetas)
    print(f"k={k}  silueta={s:.3f}  " + "█" * int(s * 50))

<details><summary><b>👀 Ver respuesta</b></summary>

El máximo está en **k = 4** (≈0.78): el mismo veredicto que el codo. Fíjate en que k = 3 y k = 5 no quedan tan lejos (≈0.67): juntar dos grupos cercanos o partir uno en dos "no está tan mal". En datos reales las diferencias suelen ser aún más pequeñas. Cuando dos valores de k están cerca, **decide el negocio**: ¿cuántas campañas distintas puede manejar el equipo de marketing?

Ahora sí, mira los datos:
</details>

In [ ]:
km4 = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X4)
plt.scatter(*X4.T, c=km4.labels_, cmap="viridis", s=12)
plt.scatter(*km4.cluster_centers_.T, c="red", marker="X", s=200, edgecolor="k")
plt.title("k = 4"); plt.show()

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Grupos que solo separan una columna | Datos **sin escalar** | `StandardScaler` antes de K-means (Lección 8.1) |
| Un centroide "arrastrado" lejos de su grupo, o un grupo de 1–2 puntos | **Atípicos**: el promedio es sensible a extremos | Revisar atípicos (8.6), transformar (logaritmo), o usar `KMedoids`/DBSCAN |
| Resultados distintos cada vez que ejecutas | Sin `random_state` y/o `n_init` bajo | Fijar semilla, subir `n_init` |
| Corta formas curvas o alargadas por la mitad | K-means asume grupos **redondos y parecidos** | DBSCAN (8.3) o `GaussianMixture` |
| Elegiste k = 10 "porque la inercia era la más baja" | La inercia **siempre** baja al subir k | Codo + silueta + criterio de negocio |
| Promedios absurdos de columnas categóricas ("0.4 de mascota") | K-means usa promedios: pensado para datos **numéricos** | Codificar con cuidado, o usar métodos para datos categóricos (k-modes) |

### 🔮 Ejercicio 9 — El cliente millonario
Tenemos 200 clientes "normales" en dos grupos claros, más **un solo** cliente con un gasto descomunal. Pedimos k = 2. ¿Qué crees que hará K-means?

✍️ *Mi apuesta:*

In [ ]:
X_cli, _ = make_blobs(n_samples=200, centers=[[2, 2], [6, 6]], cluster_std=0.6, random_state=0)
X_cli = np.vstack([X_cli, [[60, 60]]])          # 👑 el cliente millonario

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_cli)
print("Tamaño de cada grupo:", np.bincount(km.labels_))
print("Centroides:\n", km.cluster_centers_.round(1))

<details><summary><b>👀 Ver respuesta</b></summary>

¡K-means le da un **grupo entero al millonario** (tamaño 1) y mete a los otros 200 clientes en un solo grupo, juntando los dos grupos reales! Para minimizar la inercia, le "conviene" dedicarle un carrito a ese punto lejanísimo, porque su distancia al cuadrado es enorme.

Moraleja: **antes de K-means, mira y trata los atípicos** (Lección 2.2 y Lección 8.6).
</details>

---
## 7. 📋 Ficha del modelo

| | K-means |
|---|---|
| **Tipo** | No supervisado: agrupamiento |
| **Intuición** | Colocar k carritos de helado para que cada cliente camine lo menos posible |
| **Cómo aprende** | Repite: asignar cada punto al centroide más cercano → mover cada centroide al promedio de su grupo, hasta que nada cambia. Minimiza la **inercia** |
| **Hiperparámetros clave** | `n_clusters` (k), `init`, `n_init` |
| **✅ Fortalezas** | Simple y explicable (cada grupo = un "cliente promedio") · Muy rápido, escala a millones de datos (`MiniBatchKMeans`) · Asigna datos nuevos con `predict` |
| **❌ Debilidades** | Hay que elegir k · Asume grupos redondos y de tamaño parecido · Sensible a escala y atípicos · Puede caer en mínimos locales |
| **Úsalo cuando…** | Quieres segmentar datos numéricos en grupos compactos y explicarlos con su promedio |
| **Evítalo cuando…** | Los grupos tienen formas raras, hay mucho ruido o atípicos, o los datos son mayormente categóricos |

## 8. 🧠 Resumen en 5 puntos
1. K-means coloca **k centroides** y repite **asignar → mover** hasta que nada cambia.
2. Minimiza la **inercia** (qué tan lejos está cada punto de su centro); nunca sube entre iteraciones.
3. Puede caer en **mínimos locales**: usa `k-means++` y `n_init ≥ 10`.
4. Elige k con el **codo** (inercia) y la **silueta** (máximo), y confirma con el negocio.
5. Asume grupos **redondos, de tamaño parecido, escalados y sin atípicos**; si no se cumple, usa otros métodos (8.3).

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** ¿Qué es un centroide?
- a) El punto más extremo del grupo · b) El promedio de los puntos del grupo · c) El primer punto del grupo · d) Un atípico

**2.** Pasas de k = 5 a k = 6 con los mismos datos. ¿Qué le pasa a la inercia (con buena inicialización)?
- a) Sube · b) Baja o se queda igual · c) Puede pasar cualquier cosa · d) Se vuelve 0

**3.** Ejecutas K-means 5 veces con `n_init=1` y obtienes 5 inercias distintas. ¿Qué significa?
- a) Hay un error en los datos · b) Cada inicio cayó en un mínimo local distinto · c) k está mal elegido · d) Faltó escalar

**4.** Un punto tiene silueta −0.3. ¿Qué indica?
- a) Está en el centro de su grupo · b) Está justo en la frontera · c) Probablemente está en el grupo equivocado · d) Es un atípico seguro

**5.** ¿Cuál de estos datos es el **peor** candidato para K-means?
- a) Clientes con 3 grupos redondos de gasto · b) Dos espirales entrelazadas · c) Píxeles de una imagen por color · d) Tiendas por ventas y tamaño (escaladas)

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** "Means" = medias: el centroide es el promedio.
2. **b)** Un centro más nunca empeora la mejor solución posible (en el peor caso, se queda sin usar). Por eso la inercia no sirve para elegir k por sí sola.
3. **b)** Mínimos locales. Solución: `n_init` alto.
4. **c)** Está, en promedio, más cerca del grupo vecino que del suyo.
5. **b)** Las espirales no son redondas: K-means las cortará en rebanadas. (La c, por cierto, es un uso clásico: reducir los colores de una imagen a k colores.)
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** El codo de tus datos es ambiguo: podría ser k = 3 o k = 5, y las siluetas son casi iguales (0.41 y 0.40). ¿Cómo decidirías?

**B. Caso de análisis 💼** Una tienda en línea segmenta clientes con K-means usando `gasto_total` y `visitas_al_mes`. El grupo más pequeño tiene solo 3 clientes, que gastan 100 veces más que el resto. El gerente quiere lanzar una campaña especial "para ese segmento". ¿Qué le recomendarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Cuando las métricas empatan, decide el **uso**: ¿puede el equipo manejar 5 campañas distintas o solo 3? Además, perfila los grupos en ambos casos: si con k = 5 aparecen dos grupos que el negocio trataría distinto (p. ej. "jóvenes que compran de noche"), vale la pena; si solo se parte un grupo en dos casi iguales, quédate con 3. También puedes comprobar la **estabilidad**: repite con submuestras y mira qué k da grupos más consistentes.

**B.** Ese "segmento" es casi seguro un efecto de los **atípicos**: K-means le dedicó un centroide a 3 clientes extremos. No es un segmento, son 3 clientes VIP (o errores de datos). Recomendación: tratarlos aparte (atención personalizada), revisar que no sean errores, y **volver a segmentar** sin ellos o con una transformación logarítmica del gasto, para que el resto de los grupos tenga sentido.
</details>

### 9.3 🧩 Práctica integradora: "Segmentando a los clientes del centro comercial"

Un centro comercial tiene datos de 400 clientes de su tarjeta de lealtad: **ingreso anual** (miles de pesos) y **puntaje de gasto** (1–100, asignado según su historial de compras). Quiere **segmentarlos** para diseñar campañas.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(42)
centros_reales = [(25, 20), (25, 80), (55, 50), (90, 15), (90, 85)]
tamaños = [60, 60, 160, 60, 60]
filas = [rng.normal(c, (7, 7), (n, 2)) for c, n in zip(centros_reales, tamaños)]
mall = pd.DataFrame(np.vstack(filas), columns=["ingreso_miles", "puntaje_gasto"])
mall = mall.clip(lower=1).round(0)
mall["puntaje_gasto"] = mall["puntaje_gasto"].clip(upper=100)
mall = mall.sample(frac=1, random_state=0).reset_index(drop=True)   # mezclamos el orden

plt.scatter(mall.ingreso_miles, mall.puntaje_gasto, s=12)
plt.xlabel("Ingreso anual (miles)"); plt.ylabel("Puntaje de gasto"); plt.show()
mall.describe().round(1)

**Paso 2 ✋ — ¿Hay que escalar?** Las dos columnas tienen rangos parecidos (≈1–110 y 1–100). ¿Escalarías igual? ¿Por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Aquí escalar **casi no cambia nada** porque los rangos son similares. Aun así es buena costumbre escalar siempre: el día que alguien agregue "número de visitas" (0–5) o "gasto en pesos" (0–1,000,000) el análisis no se rompe en silencio. Lo haremos dentro del código.
</details>

**Paso 3 🔮 — Elige k.** Mirando la gráfica del Paso 1, ¿cuántos grupos ves? Luego ejecuta el codo y la silueta.

✍️ *Mi apuesta:*

In [ ]:
X_mall = StandardScaler().fit_transform(mall)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
K = range(2, 11)
modelos = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_mall) for k in K]
ax1.plot(K, [m.inertia_ for m in modelos], "o-"); ax1.set_title("Codo (inercia)"); ax1.set_xlabel("k")
siluetas = [silhouette_score(X_mall, m.labels_) for m in modelos]
ax2.plot(K, siluetas, "s-", color="C1"); ax2.set_title("Silueta promedio"); ax2.set_xlabel("k")
plt.show()
print("Mejor k por silueta:", K[int(np.argmax(siluetas))], f"(silueta = {max(siluetas):.2f})")

<details><summary><b>👀 Ver respuesta</b></summary>

Se ven **5 grupos**: cuatro esquinas y uno en el centro. El codo se dobla en k = 5 y la silueta tiene su máximo en **k = 5** (≈0.71). Las tres señales (vista, codo y silueta) coinciden.
</details>

**Paso 4 ▶️ — Agrupa y perfila.** Entrenamos con k = 5 y calculamos el **perfil** de cada grupo: cuántos clientes tiene y su promedio en unidades originales (no escaladas, que es lo que entiende el negocio).

In [ ]:
K_ELEGIDO = 5     # ✏️
km_mall = KMeans(n_clusters=K_ELEGIDO, n_init=10, random_state=0).fit(X_mall)
mall["segmento"] = km_mall.labels_

perfil = mall.groupby("segmento").agg(clientes=("ingreso_miles", "size"),
                                      ingreso=("ingreso_miles", "mean"),
                                      gasto=("puntaje_gasto", "mean")).round(0)
print(perfil)
plt.scatter(mall.ingreso_miles, mall.puntaje_gasto, c=mall.segmento, cmap="tab10", s=12)
for seg, fila in perfil.iterrows():
    plt.annotate(str(seg), (fila.ingreso, fila.gasto), fontsize=16, weight="bold")
plt.xlabel("Ingreso anual (miles)"); plt.ylabel("Puntaje de gasto"); plt.show()

**Paso 5 ✋ — Ponles nombre.** Un número de grupo no le dice nada a marketing. Dale a cada segmento un **nombre** y una **acción** de campaña.

✍️ *Mis segmentos:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los números de segmento pueden variar, pero los perfiles deberían ser estos:

| Perfil | Nombre posible | Acción |
|---|---|---|
| Ingreso alto (~90), gasto alto (~85) | 💎 **Premium** | Programa VIP, lanzamientos exclusivos |
| Ingreso alto (~90), gasto bajo (~15) | 🔒 **Ahorradores pudientes** | Campañas de calidad/valor para "despertar" su gasto: el mayor potencial |
| Ingreso medio (~55), gasto medio (~50) | 🙂 **Clientes promedio** (el grupo más grande, ~160) | Campañas generales, fidelización |
| Ingreso bajo (~25), gasto alto (~80) | 🛍️ **Entusiastas** | Ofertas y meses sin intereses (cuidado con el riesgo de crédito) |
| Ingreso bajo (~25), gasto bajo (~20) | 🎯 **Sensibles al precio** | Descuentos y productos básicos |

Lo valioso no es el algoritmo, sino esta tabla: **convierte grupos en decisiones**.
</details>

**Paso 6 🔮 — Un cliente nuevo.** Llega una clienta con ingreso de 85 mil y puntaje de gasto 20. ¿A qué segmento irá? Predícelo con tu tabla y comprueba. (K-means puede asignar datos nuevos con `predict`: basta con ver cuál centroide está más cerca.)

✍️ *Mi apuesta:*

In [ ]:
nueva = pd.DataFrame({"ingreso_miles": [85], "puntaje_gasto": [20]})
escalador_mall = StandardScaler().fit(mall[["ingreso_miles", "puntaje_gasto"]])
seg = km_mall.predict(escalador_mall.transform(nueva))[0]
print(f"Segmento asignado: {seg}")
print(perfil.loc[seg])

<details><summary><b>👀 Ver respuesta</b></summary>

Va al segmento de **ingreso alto y gasto bajo** ("ahorradores pudientes"): su centroide es el más cercano. Fíjate en que tuvimos que **escalarla con el mismo escalador** que usamos para entrenar; si no, el `predict` compararía peras con manzanas (Lección 7.5: por eso en la práctica todo va en un `Pipeline`).
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Visualizing K-Means Clustering* (Naftali Harris): mueve los centroides y mira cada iteración.
- 📺 **Video:** StatQuest, "K-means clustering" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Clustering → K-means* y *Gaussian mixture models* (para grupos alargados o de distinto tamaño).

---
⬅️ **Anterior:** [8.1 Encontrar estructura sin etiquetas](8.1-estructura-sin-etiquetas.ipynb) · ➡️ **Siguiente:** [8.3 Clustering jerárquico y DBSCAN: árboles genealógicos y multitudes](8.3-jerarquico-y-dbscan.ipynb)